In [ ]:
# Bird Counter - Jupyter Notebook Version
# This version is optimized for running in a Jupyter notebook

import cv2
import numpy as np
from collections import defaultdict
import math
import tkinter as tk
from tkinter import filedialog, messagebox
from datetime import datetime
import os
import time
import multiprocessing

# Configuration settings (defined at global scope for notebook compatibility)
FRAME_COVERAGE_PERCENTAGE = 0.6
FRAME_SKIP = 1  # Process every Nth frame (higher = faster but less accurate)
RESIZE_FACTOR = 0.9  # Resize frame by this factor (smaller = faster)
ENABLE_MULTIPROCESSING = False  # Disabled for notebook compatibility
MAX_WORKERS = max(1, multiprocessing.cpu_count() - 1)  # Use all but one CPU core

# Track previous frames to detect movement
prev_frame = None
motion_threshold = 30  # Adjust based on your needs

# Add profile decorator for performance debugging
def profile(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@profile
def detect_moving_birds(current_detections, frame, prev_frame_cache):
    """Detect moving birds based on frame differencing - optimized version"""
    global prev_frame
    
    # Use cached frame if available
    if prev_frame_cache is not None:
        prev_gray = prev_frame_cache
    elif prev_frame is None:
        prev_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return [], prev_frame
    else:
        prev_gray = prev_frame
    
    # Convert current frame to grayscale
    current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Calculate frame difference
    frame_diff = cv2.absdiff(prev_gray, current_gray)
    
    # Vectorized operation for all detections at once
    moving_birds = []
    if current_detections:
        for detection in current_detections:
            x, y, w, h = detection['bbox']
            # Ensure within bounds
            y2 = min(y+h, frame_diff.shape[0])
            x2 = min(x+w, frame_diff.shape[1])
            
            # Check if there's significant motion in the detection area
            if y < y2 and x < x2:  # Ensure valid ROI
                roi_diff = frame_diff[y:y2, x:x2]
                if roi_diff.size > 0:  # Make sure ROI is not empty
                    motion_score = np.mean(roi_diff)
                    
                    if motion_score > motion_threshold:
                        moving_birds.append(detection)
    
    return moving_birds, current_gray

class EnhancedBirdTracker:
    def __init__(self, max_stationary_frames=20, min_movement_distance=15, min_flight_duration=5):
        self.tracks = {}
        self.track_id = 0
        self.max_stationary_frames = max_stationary_frames
        self.min_movement_distance = min_movement_distance
        self.min_flight_duration = min_flight_duration
        
        # Track unique flying birds
        self.confirmed_flying_birds = set()  # Set of track IDs that have been confirmed as flying
        self.bird_flight_status = {}  # Track flight status for each bird
    
    @profile
    def update_tracks(self, detections):
        """Update bird tracks - optimized with spatial hashing"""
        # Match detections to existing tracks
        matched_tracks = {}
        current_frame_birds = []
        
        # Create a spatial hash of current tracks for faster matching
        active_tracks = {}
        for track_id, track_data in self.tracks.items():
            active_tracks[track_id] = track_data['positions'][-1]
        
        for detection in detections:
            x, y = detection['center']
            best_match = None
            min_distance = float('inf')
            
            # Use vectorized operations for distance calculation if many tracks
            if len(active_tracks) > 50:  # Only worth vectorizing for many tracks
                track_ids = list(active_tracks.keys())
                positions = np.array(list(active_tracks.values()))
                
                # Vectorized distance calculation
                if len(positions) > 0:
                    distances = np.sqrt(((positions[:, 0] - x) ** 2) + ((positions[:, 1] - y) ** 2))
                    min_idx = np.argmin(distances)
                    if distances[min_idx] < 100:  # Distance threshold
                        min_distance = distances[min_idx]
                        best_match = track_ids[min_idx]
            else:
                # Standard approach for fewer tracks
                for track_id, last_pos in active_tracks.items():
                    if track_id not in matched_tracks:
                        dx = x - last_pos[0]
                        dy = y - last_pos[1]
                        distance = math.sqrt(dx*dx + dy*dy)
                        if distance < min_distance and distance < 100:
                            min_distance = distance
                            best_match = track_id
            
            if best_match:
                # Update existing track
                self.tracks[best_match]['positions'].append((x, y))
                self.tracks[best_match]['stationary_count'] = 0
                self.tracks[best_match]['last_seen'] = len(self.tracks[best_match]['positions'])
                matched_tracks[best_match] = detection
                
                # Check if this bird qualifies as flying
                if self.is_bird_flying(best_match):
                    self.confirmed_flying_birds.add(best_match)
                    self.bird_flight_status[best_match] = 'flying'
                    current_frame_birds.append(detection)
                
            else:
                # Create new track
                self.tracks[self.track_id] = {
                    'positions': [(x, y)],
                    'stationary_count': 0,
                    'last_seen': 1,
                    'created_frame': len(list(self.tracks.values())[0]['positions']) if self.tracks else 0
                }
                self.bird_flight_status[self.track_id] = 'new'
                self.track_id += 1
        
        # Update unmatched tracks (increment stationary count)
        tracks_to_remove = []
        for track_id in list(self.tracks.keys()):
            if track_id not in matched_tracks:
                self.tracks[track_id]['stationary_count'] += 1
                if self.tracks[track_id]['stationary_count'] >= self.max_stationary_frames:
                    tracks_to_remove.append(track_id)
        
        # Batch remove obsolete tracks
        for track_id in tracks_to_remove:
            del self.tracks[track_id]
            if track_id in self.bird_flight_status:
                del self.bird_flight_status[track_id]
        
        return current_frame_birds
    
    @profile
    def is_bird_flying(self, track_id):
        """Determine if a bird is actively flying based on movement patterns - optimized"""
        positions = self.tracks[track_id]['positions']
        
        # Need minimum number of positions to determine flight
        if len(positions) < self.min_flight_duration:
            return False
        
        # Convert to numpy array for faster calculations
        pos_array = np.array(positions[-10:])  # Only use last 10 positions for efficiency
        
        if len(pos_array) < 2:
            return False
            
        # Calculate distances between consecutive points
        diffs = np.diff(pos_array, axis=0)
        distances = np.sqrt(np.sum(diffs**2, axis=1))
        
        # Calculate total distance and average movement
        total_distance = np.sum(distances)
        avg_movement = total_distance / len(distances) if len(distances) > 0 else 0
        
        # Calculate movement consistency using vectorized operations
        movement_consistency = 0
        if len(diffs) >= 2:
            # Normalize direction vectors
            norms = np.sqrt(np.sum(diffs**2, axis=1))
            valid_idx = norms > 0
            if np.sum(valid_idx) >= 2:
                norm_diffs = diffs[valid_idx] / norms[valid_idx, np.newaxis]
                
                # Calculate dot products between pairs of directions
                dot_products = np.abs(np.dot(norm_diffs, norm_diffs.T))
                # Exclude self-comparisons (diagonal)
                mask = ~np.eye(len(norm_diffs), dtype=bool)
                movement_consistency = np.mean(dot_products[mask]) if np.sum(mask) > 0 else 0
        
        # A bird is considered flying if it meets movement criteria
        is_flying = (
            total_distance > self.min_movement_distance and 
            avg_movement > 2.0 and  # Average 2+ pixels movement per frame
            movement_consistency > 0.3  # Reasonably consistent movement
        )
        
        return is_flying
    
    def get_unique_flying_birds_count(self):
        """Get the total number of unique birds that have been confirmed as flying"""
        return len(self.confirmed_flying_birds)
    
    def get_currently_active_birds(self):
        """Get the number of birds currently being tracked and flying"""
        currently_flying = 0
        for track_id in self.tracks:
            if track_id in self.confirmed_flying_birds:
                currently_flying += 1
        return currently_flying

@profile
def detect_birds_in_frame(frame, frame_height=None, detection_boundary=None):
    """Optimized bird detection with precomputed values and region-of-interest focus"""
    # Calculate boundaries once if not provided
    if frame_height is None:
        frame_height = frame.shape[0]
    
    if detection_boundary is None:
        detection_boundary = int(frame_height * FRAME_COVERAGE_PERCENTAGE)
    
    # Create region of interest (top portion of frame only)
    roi = frame[:detection_boundary, :]
    
    # Convert to grayscale (region of interest only)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise (smaller kernel for speed)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    
    # Apply adaptive threshold to detect dark objects (birds) against sky
    # Use a larger block size for speedadaptive thresholding 
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 15, 2)
    
    # Morphological operations to clean up the binary image (combined for efficiency)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    # Find contours (using simpler method)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Pre-filter contours by area for efficiency
    detections = []
    for contour in contours:
        # Calculate contour area (fast calculation)
        area = cv2.contourArea(contour)
        
        # Filter by area (adjusted for better performance)
        if 10 < area < 1000:  # Area thresholds depend on resolution
            x, y, w, h = cv2.boundingRect(contour)
            
            # Filter by aspect ratio (birds are usually not perfect circles)
            aspect_ratio = w / h if h > 0 else 0
            if 0.3 < aspect_ratio < 4.0:
                center_x = x + w // 2
                center_y = y + h // 2
                
                detection = {
                    'bbox': (x, y, w, h),
                    'center': (center_x, center_y),
                    'area': area,
                    'confidence': min(1.0, area / 100.0)  # Simple confidence based on size
                }
                detections.append(detection)
    
    return detections

@profile
def process_video_with_unique_bird_counting(video_path, show_display=False):
    """Optimized video processing with frame skipping and resolution reduction"""
    global prev_frame
    prev_frame = None
    prev_frame_cache = None
    
    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Calculate new dimensions for resizing (maintain aspect ratio)
    new_width = int(width * RESIZE_FACTOR)
    new_height = int(height * RESIZE_FACTOR)
    
    # Calculate detection boundary once
    detection_boundary = int(new_height * FRAME_COVERAGE_PERCENTAGE)
    
    print(f"Processing: {os.path.basename(video_path)}")
    print(f"- Original Resolution: {width}x{height}, Resized: {new_width}x{new_height}")
    print(f"- FPS: {fps}, Frames: {total_frames}, Processing every {FRAME_SKIP} frames")
    print(f"- Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}% of frame")
    
    if show_display:
        print(f"Processing... Press 'q' to quit, 'p' to pause/resume, 's' to step frame by frame")
    
    # Initialize enhanced tracking with adjusted parameters
    bird_tracker = EnhancedBirdTracker(
        max_stationary_frames=int(30 / FRAME_SKIP),  # Adjust for frame skipping
        min_movement_distance=25 * RESIZE_FACTOR,   # Adjust for resizing
        min_flight_duration=int(8 / FRAME_SKIP)     # Adjust for frame skipping
    )
    
    # Initialize background subtractor
    back_sub = cv2.createBackgroundSubtractorMOG2(detectShadows=False)  # Disable shadows for speed
    
    # Statistics
    frame_count = 0
    max_concurrent_birds = 0
    paused = False
    step_mode = False
    processed_frames = 0
    
    # Performance tracking
    start_time = time.time()
    last_update_time = start_time
    
    try:
        while True:
            if not paused or step_mode:
                # Skip frames for performance
                for _ in range(FRAME_SKIP - 1):
                    ret = cap.grab()  # Just grab frame without decoding
                    if not ret:
                        break
                
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame_count += FRAME_SKIP  # Count skipped frames
                processed_frames += 1
                
                # Resize frame for faster processing
                if RESIZE_FACTOR != 1.0:
                    frame = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
                
                # Apply background subtraction (only to ROI for speed)
                roi = frame[:detection_boundary, :]
                fg_mask = back_sub.apply(roi)
                
                # Use cached and optimized calculations
                current_detections = detect_birds_in_frame(frame, new_height, detection_boundary)
                
                # Apply motion filtering
                moving_detections, prev_frame_cache = detect_moving_birds(current_detections, frame, prev_frame_cache)
                
                # Update enhanced bird tracker
                currently_flying_birds = bird_tracker.update_tracks(moving_detections)
                
                # Get statistics
                unique_flying_birds = bird_tracker.get_unique_flying_birds_count()
                currently_active = bird_tracker.get_currently_active_birds()
                max_concurrent_birds = max(max_concurrent_birds, currently_active)
                
                # Calculate FPS every second
                current_time = time.time()
                elapsed = current_time - last_update_time
                if elapsed >= 1.0:
                    fps_estimate = processed_frames / elapsed
                    processed_frames = 0
                    last_update_time = current_time
                
                # For batch processing, only show display if requested
                if show_display:
                    display_frame = frame.copy()
                    
                    # Draw visualization boundary
                    cv2.line(display_frame, (0, detection_boundary), (frame.shape[1], detection_boundary), 
                            (255, 255, 0), 2)
                    
                    # Add text to show the detection area
                    cv2.putText(display_frame, f"Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}%", 
                               (10, detection_boundary + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                    
                    # Only draw currently flying birds (not all detections)
                    for detection in currently_flying_birds:
                        x, y, w, h = detection['bbox']
                        cv2.rectangle(display_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
                        cv2.putText(display_frame, "FLYING", (x, y - 5),
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                    
                    # Display enhanced information
                    info_y = 30
                    cv2.rectangle(display_frame, (5, 5), (480, 155), (0, 0, 0), -1)
                    cv2.rectangle(display_frame, (5, 5), (480, 155), (57, 255, 20), 2)
                    cv2.putText(display_frame, f"Frame: {frame_count}/{total_frames}", 
                               (10, info_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 2)
                    
                    cv2.putText(display_frame, f"UNIQUE Flying Birds: {unique_flying_birds}", 
                               (10, info_y + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                    
                    cv2.putText(display_frame, f"Currently Active: {currently_active}", 
                               (10, info_y + 60), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 220, 100), 2)
                    
                    cv2.putText(display_frame, f"Max Concurrent: {max_concurrent_birds}", 
                               (10, info_y + 90), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 220, 100), 2)
                    
                    # Progress bar
                    progress = frame_count / total_frames
                    bar_width = 400
                    bar_height = 20
                    bar_x = display_frame.shape[1] - bar_width - 10
                    bar_y = 10
                    
                    cv2.rectangle(display_frame, (bar_x, bar_y), 
                                 (bar_x + bar_width, bar_y + bar_height), (100, 100, 100), -1)
                    cv2.rectangle(display_frame, (bar_x, bar_y), 
                                 (bar_x + int(bar_width * progress), bar_y + bar_height), (0, 255, 0), -1)
                    cv2.putText(display_frame, f"{progress*100:.1f}%", 
                               (bar_x + bar_width//2 - 20, bar_y + 15),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    
                    # Display frame
                    cv2.imshow("Optimized Bird Counter - Flying Birds Detection", display_frame)
                    
                    # Handle keyboard input
                    key = cv2.waitKey(1 if not paused else 0) & 0xFF
                    
                    if key == ord('q'):
                        break
                    elif key == ord('p'):
                        paused = not paused
                        step_mode = False
                    elif key == ord('s'):
                        step_mode = True
                        paused = False
                    elif key == 27:  # ESC
                        break
                    
                    if step_mode:
                        paused = True
                        step_mode = False
                
                # Print progress less frequently for batch processing
                if frame_count % (100 * FRAME_SKIP) == 0:
                    progress = frame_count / total_frames
                    elapsed_total = time.time() - start_time
                    remaining = (elapsed_total / progress) - elapsed_total if progress > 0 else 0
                    
                    print(f"  Progress: {frame_count}/{total_frames} frames ({progress*100:.1f}%) - "
                          f"Unique flying birds: {unique_flying_birds} - "
                          f"Est. remaining: {remaining:.1f}s")
        
        # Get final count
        unique_flying_birds = bird_tracker.get_unique_flying_birds_count()
        
        # Calculate processing time
        total_time = time.time() - start_time
        frames_per_second = frame_count / total_time if total_time > 0 else 0
        
        # Display final results for this video
        print(f"  COMPLETED - Unique flying birds: {unique_flying_birds}")
        print(f"  Max concurrent: {max_concurrent_birds}, Total tracks: {bird_tracker.track_id}")
        print(f"  Processing time: {total_time:.2f}s, Speed: {frames_per_second:.2f} frames/s")
        
        return {
            'video_path': video_path,
            'video_name': os.path.basename(video_path),
            'frames_processed': frame_count,
            'unique_flying_birds': unique_flying_birds,
            'max_concurrent_birds': max_concurrent_birds,
            'total_tracks': bird_tracker.track_id,
            'fps': fps,
            'total_frames': total_frames,
            'duration_seconds': total_frames/fps if fps > 0 else 0,
            'processing_time': total_time,
            'processing_speed': frames_per_second
        }
        
    except Exception as e:
        print(f"  Error processing video: {str(e)}")
        return None
        
    finally:
        # Cleanup
        cap.release()
        if show_display:
            cv2.destroyAllWindows()

def write_results_to_file(results, output_file_path):
    """Write bird counting results to a text file"""
    try:
        with open(output_file_path, 'w', encoding='utf-8') as f:
            f.write("BIRD COUNTING RESULTS (OPTIMIZED VERSION)\n")
            f.write("=" * 80 + "\n")
            f.write(f"Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}% of frame\n")
            f.write(f"Frame Skip: {FRAME_SKIP} (processing every {FRAME_SKIP} frames)\n")
            f.write(f"Resize Factor: {RESIZE_FACTOR:.2f}\n")
            f.write(f"Total Videos Processed: {len([r for r in results if r is not None])}\n")
            f.write(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 80 + "\n\n")
            
            # Summary statistics
            successful_results = [r for r in results if r is not None]
            if successful_results:
                total_birds = sum(r['unique_flying_birds'] for r in successful_results)
                avg_birds = total_birds / len(successful_results)
                max_birds = max(r['unique_flying_birds'] for r in successful_results)
                min_birds = min(r['unique_flying_birds'] for r in successful_results)
                total_processing_time = sum(r['processing_time'] for r in successful_results)
                
                f.write("SUMMARY STATISTICS:\n")
                f.write("-" * 40 + "\n")
                f.write(f"Total Birds Detected Across All Videos: {total_birds}\n")
                f.write(f"Average Birds per Video: {avg_birds:.1f}\n")
                f.write(f"Maximum Birds in Single Video: {max_birds}\n")
                f.write(f"Minimum Birds in Single Video: {min_birds}\n")
                f.write(f"Total Processing Time: {total_processing_time:.2f} seconds\n")
                f.write("\n")
            
            # Individual video results
            f.write("INDIVIDUAL VIDEO RESULTS:\n")
            f.write("-" * 40 + "\n")
            
            for i, result in enumerate(results, 1):
                if result is not None:
                    f.write(f"{i:2d}. {result['video_name']}\n")
                    f.write(f"    Unique Flying Birds: {result['unique_flying_birds']}\n")
                    f.write(f"    Max Concurrent Birds: {result['max_concurrent_birds']}\n")
                    f.write(f"    Total Tracks Created: {result['total_tracks']}\n")
                    f.write(f"    Video Duration: {result['duration_seconds']:.1f} seconds\n")
                    f.write(f"    Frames Processed: {result['frames_processed']}/{result['total_frames']}\n")
                    f.write(f"    Processing Time: {result['processing_time']:.2f} seconds\n")
                    f.write(f"    Processing Speed: {result['processing_speed']:.2f} frames/second\n")
                    f.write(f"    File Path: {result['video_path']}\n")
                    f.write("\n")
                else:
                    f.write(f"{i:2d}. [FAILED TO PROCESS]\n\n")
            
            # CSV-style summary for easy import into spreadsheets
            f.write("\nCSV FORMAT (for spreadsheet import):\n")
            f.write("-" * 40 + "\n")
            f.write("Video_Name,Unique_Flying_Birds,Max_Concurrent,Total_Tracks,Duration_Seconds,Processing_Time,Speed_FPS\n")
            
            for result in results:
                if result is not None:
                    f.write(f"{result['video_name']},{result['unique_flying_birds']},"
                           f"{result['max_concurrent_birds']},{result['total_tracks']},"
                           f"{result['duration_seconds']:.1f},{result['processing_time']:.2f},"
                           f"{result['processing_speed']:.2f}\n")
        
        print(f"\nResults written to: {output_file_path}")
        return True
        
    except Exception as e:
        print(f"Error writing results to file: {str(e)}")
        return False

def select_video_folder():
    """Open a dialog to select a folder containing video files"""
    root = tk.Tk()
    root.withdraw()
    
    folder_path = filedialog.askdirectory(
        title="Select Folder Containing Video Files"
    )
    
    root.destroy()
    return folder_path

def get_video_files(folder_path):
    """Get all video files in the specified folder"""
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm', '.m4v', '.mpg', '.mpeg']
    video_files = []
    
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        if os.path.isfile(file_path):
            _, ext = os.path.splitext(file_path)
            if ext.lower() in video_extensions:
                video_files.append(file_path)
    
    return video_files

def process_single_video():
    """Process a single video with display"""
    root = tk.Tk()
    root.withdraw()
    
    filetypes = [
        ("Video files", "*.mp4 *.avi *.mov *.mkv *.wmv *.flv *.webm"),
        ("All files", "*.*")
    ]
    
    video_path = filedialog.askopenfilename(
        title="Select Video File for Bird Counting",
        filetypes=filetypes
    )
    root.destroy()
    
    if not video_path:
        print("No video file selected. Exiting...")
        return
    
    print(f"Selected video: {os.path.basename(video_path)}")
    
    try:
        result = process_video_with_unique_bird_counting(video_path, show_display=True)
        if result:
            print(f"\n{'='*60}")
            print(f"FINAL RESULTS")
            print(f"{'='*60}")
            print(f"Video: {result['video_name']}")
            print(f"TOTAL UNIQUE FLYING BIRDS: {result['unique_flying_birds']}")
            print(f"Maximum concurrent flying birds: {result['max_concurrent_birds']}")
            print(f"Processing time: {result['processing_time']:.2f} seconds")
            print(f"Processing speed: {result['processing_speed']:.2f} frames/second")
            print(f"{'='*60}")
    except Exception as e:
        print(f"Error processing video: {str(e)}")
        messagebox.showerror("Error", f"Failed to process video:\n{str(e)}")

def process_batch_videos():
    """Process multiple videos in batch mode"""
    # Select folder containing videos
    folder_path = select_video_folder()
    
    if not folder_path:
        print("No folder selected. Exiting...")
        return
    
    # Get all video files in the folder
    video_files = get_video_files(folder_path)
    
    if not video_files:
        print(f"No video files found in folder: {folder_path}")
        print("Supported formats: .mp4, .avi, .mov, .mkv, .wmv, .flv, .webm, .m4v, .mpg, .mpeg")
        return
    
    print(f"\nFound {len(video_files)} video files in: {folder_path}")
    print("Video files to process:")
    for i, video_file in enumerate(video_files, 1):
        print(f"  {i:2d}. {os.path.basename(video_file)}")
    
    # Confirm processing
    try:
        confirm = input(f"\nProcess all {len(video_files)} videos? (y/n): ").strip().lower()
        if confirm not in ['y', 'yes']:
            print("Processing cancelled.")
            return
    except KeyboardInterrupt:
        print("\nProcessing cancelled.")
        return
    
    # Process all videos
    results = []
    total_birds_all_videos = 0
    successful_count = 0
    failed_count = 0
    
    print(f"\n{'='*60}")
    print(f"STARTING BATCH PROCESSING")
    print(f"{'='*60}")
    
    start_time = datetime.now()
    
    # Process videos sequentially (multiprocessing disabled for notebooks)
    for i, video_path in enumerate(video_files, 1):
        print(f"\n[{i}/{len(video_files)}] Processing: {os.path.basename(video_path)}")
        print("-" * 50)
        
        try:
            result = process_video_with_unique_bird_counting(video_path, show_display=False)
            
            if result is not None:
                results.append(result)
                total_birds_all_videos += result['unique_flying_birds']
                successful_count += 1
                print(f"  SUCCESS - {result['unique_flying_birds']} birds detected in {result['processing_time']:.2f}s")
            else:
                results.append(None)
                failed_count += 1
                print(f"  FAILED - Could not process video")
                
        except Exception as e:
            print(f"  ERROR - {str(e)}")
            results.append(None)
            failed_count += 1
    
    end_time = datetime.now()
    processing_time = end_time - start_time
    
    # Generate output file path
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    folder_name = os.path.basename(folder_path)
    output_filename = f"bird_count_results_{folder_name}_{timestamp}.txt"
    output_path = os.path.join(folder_path, output_filename)
    
    # Write results to file
    write_results_to_file(results, output_path)
    
    # Display final summary
    print(f"\n{'='*60}")
    print(f"BATCH PROCESSING COMPLETED")
    print(f"{'='*60}")
    print(f"Total Videos: {len(video_files)}")
    print(f"Successfully Processed: {successful_count}")
    print(f"Failed: {failed_count}")
    print(f"Total Processing Time: {processing_time}")
    print(f"TOTAL BIRDS DETECTED ACROSS ALL VIDEOS: {total_birds_all_videos}")
    if successful_count > 0:
        print(f"Average Birds per Video: {total_birds_all_videos/successful_count:.1f}")
    print(f"Results saved to: {output_filename}")
    print(f"{'='*60}")

# This function is meant to be called directly from a notebook cell
def run_bird_counter():
    """Main entry point for the bird counter application"""
    print("Optimized Bird Counter - Jupyter Notebook Version")
    print("=" * 60)
    print(f"Detection Area: Top {FRAME_COVERAGE_PERCENTAGE*100:.1f}% of frame")
    print(f"Frame Skip: {FRAME_SKIP} (processing every {FRAME_SKIP} frames)")
    print(f"Resize Factor: {RESIZE_FACTOR:.2f}")
    print("Multiprocessing: Disabled (for notebook compatibility)")
    
    print("\nTo change settings, modify the constants at the top of the notebook")
    print()
    
    # Ask user for processing mode
    print("Processing Options:")
    print("1. Batch process all videos in a folder (no display)")
    print("2. Process single video with display")
    
    while True:
        try:
            choice = input("Enter your choice (1 or 2): ").strip()
            if choice in ['1', '2']:
                break
            else:
                print("Please enter 1 or 2")
        except KeyboardInterrupt:
            print("\nExiting...")
            return
    
    if choice == '2':
        process_single_video()
    else:
        process_batch_videos()

#Takes video properties -> calculates new dimensions for resizing -> adjust the minimum flight duration, the number of stationary frames -> subtract the background -> start timer -> detect_birds_inFrame() ----> calculate boundaries for only the region of interest -> convert to gray -> gaussian blur -> adaptive thresholding -> morphological operations -> finding contours -> filter contours and draw bounding rectangles -> detect_moving_birds(observed_birds) --- -> convert frame to gray -> calculate frame difference -> check if there is significant motion -> update moving birds in the video -> get the count of the total unique birds.


: 

In [ ]:
run_bird_counter()

Optimized Bird Counter - Jupyter Notebook Version
Detection Area: Top 60.0% of frame
Frame Skip: 1 (processing every 1 frames)
Resize Factor: 0.90
Multiprocessing: Disabled (for notebook compatibility)

To change settings, modify the constants at the top of the notebook

Processing Options:
1. Batch process all videos in a folder (no display)
2. Process single video with display
